# Phase 1: การเตรียมข้อมูล (Data Preprocessing)
ในส่วนนี้เราจะทำการโหลดข้อมูล เปลี่ยนชื่อคอลัมน์ให้ใช้งานง่ายขึ้น และสร้าง Target Variable (`risk_level`) โดยใช้ระบบ Risk Scoring ที่ประเมินจาก สภาพคล่อง, ประวัติการหมุนเงินไม่ทัน, และภาระหนี้สิน

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ตั้งค่าฟอนต์ภาษาไทยสำหรับกราฟ
plt.rcParams['font.family'] = 'Tahoma'

# 1. โหลดข้อมูล
df = pd.read_csv('income_expense_behavior_vs_cost_of_living_crisis.csv')
print(f'ขนาดของข้อมูลเริ่มต้น: {df.shape}')
df.head(2)

In [ ]:
# 2. เปลี่ยนชื่อคอลัมน์ภาษาไทยให้เป็นภาษาอังกฤษ
column_mapping = {
    "1. เพศของท่าน": "sex",
    "2. ช่วงอายุเเละช่วงวัย": "age_group",
    "3. สถานะการทำงานของท่าน": "occupation",
    "4. รายได้สุทธิเฉลี่ยต่อเดือน (หลังหักภาษี)": "income",
    "5. จำนวนสมาชิกในครอบครัวที่อยู่ในความดูเเลรับผิดชอบด้านค่าใช้จ่าย (รวมตัวท่าน) ": "family_size",
    "6. ภาระหนี้สินคงที่ต่อเดือน": "debt",
    "7. ค่าใช้จ่ายด้านน้ำมันเเละการเดินทางเฉลี่ยต่อเดือน (บาท)": "travel_cost",
    "8. จำนวนผลิตภัณฑ์บัตรเครดิตหรือวงเงินสินเชื่อหมุนเวียนที่ถือครอง": "credit_cards",
    "9. ลักษณะการชำระหนี้บัตรเครดิตในรอบ 6 เดือนที่ผ่านมา": "payment_behavior",
    " 10. ความถี่ในการเลือกซื้อสินค้าตามความพึงพอใจ โดยไม่ได้วางเเผน": "impulse_buying",
    "11. การถือครองทรัพย์สินหรือการลงทุน ": "assets",
    "12. สภาพคล่องคงเหลือ (กรณีรายได้หยุด ชะงัก ท่านมีเงินสำรองอยู่ได้กี่เดือน?)": "liquidity",
    "13. ในรอบ 1 ปีที่ผ่านมา ท่านเคยประสบสภาวะ \"หมุนเงินไม่ทัน\" จนต้องหยิบยืมเงินหรือไม่": "shortage",
    "14. หากเกิดเหตุฉุกเฉินที่ต้องใช้เงินก้อนทันที ท่านมีเเหล่งเงินทุนรองรับหรือไม่": "emergency_fund",
    "15. ท่านมีความกังวลต่อสถานะทางการเงินของตนเองในปัจจุบันระดับใด": "concern_level",
    "16. หากราคาน้ำมันเเละค่าครองชีพสูงขึ้น ร้อยละ 20 ท่านจะให้ความสำคัญกับการดำเนินการสิ่งใดมากที่สุด": "coping_strategy",
    "17. ท่านมีความรู้ความเข้าใจด้านการวางเเผนการเงินเเละภาษี": "financial_literacy",
    "18. ท่านมีสวัสดิการหรือประกันสุขภาพส่วนบุคคลที่เพียงพอต่อความเสี่ยง ในรูปแบบใดบ้าง ": "welfare"
}
df = df.rename(columns=column_mapping)
print("เปลี่ยนชื่อคอลัมน์เรียบร้อยแล้ว")

In [ ]:
# แปลงกลุ่มอายุ (Generation) และรายได้เป็นตัวเลข เพื่อให้ดูกราฟง่ายขึ้น
age_map = {
    'Gen Z (เกิดปี พ.ศ. 2541 - 2555 / อายุ 14 - 28 ปี)': 'Gen Z',
    'Gen Y / Millennials (เกิดปี พ.ศ. 2523 - 2540 / อายุ 29 - 46 ปี)': 'Gen Y',
    'Gen X (เกิดปี พ.ศ. 2508 - 2522 / อายุ 47 - 61 ปี)': 'Gen X'
}
df['age_group'] = df['age_group'].map(age_map)

income_map = {
    'ต่ำกว่า 10,000 บาท': 5000,
    '10,001 - 25,000 บาท': 17500,
    '25,001 - 50,000 บาท': 37500,
    'มากกว่า 50,000 บาท': 75000
}
df['income_num'] = df['income'].map(income_map)

In [ ]:
# 3. สร้าง Target Variable (risk_level) โดยใช้ระบบคะแนน (Risk Score)

liq_score_map = {
    'น้อยกว่า 1 เดือน': 3,
    '1 - 3 เดือน': 2,
    '3 - 6 เดือน': 1,
    'มากกว่า 6 เดือน': 0
}
short_score_map = {
    'มากกว่า 2 ครั้ง': 3,
    '1 - 2 ครั้งต่อปี': 2,
    'ไม่เคย': 0
}
debt_score_map = {
    'มากกว่าร้อยละ 50 ของรายได้': 3,
    'ร้อยละ 31 - 50 ของรายได้': 2,
    'ต่ำกว่าร้อยละ 30 ของรายได้': 1,
    'ไม่มีภาระหนี้สิน': 0
}

df['liquidity_score'] = df['liquidity'].map(liq_score_map)
df['shortage_score'] = df['shortage'].map(short_score_map)
df['debt_score'] = df['debt'].map(debt_score_map)

# คำนวณคะแนนรวม (เต็ม 9)
df['total_risk_score'] = df['liquidity_score'] + df['shortage_score'] + df['debt_score']

# ตัดเกรดความเสี่ยง
def calculate_risk(score):
    if score >= 6:
        return 'High Risk'
    elif score >= 3:
        return 'Medium Risk'
    else:
        return 'Low Risk'

df['risk_level'] = df['total_risk_score'].apply(calculate_risk)

print('สรุปจำนวนตัวอย่างในแต่ละกลุ่มความเสี่ยง:')
print(df['risk_level'].value_counts())


---
# Phase 2: การวิเคราะห์ Insight และการแสดงผล (EDA & Visualization)
เราจะใช้คำสั่งพื้นฐานของ pandas (`.plot`) และ matplotlib (`plt`) ในการสร้างกราฟเพื่อให้เข้าใจง่าย

In [ ]:
# ---------------------------------------------------------
# Insight 1: ใครคือหนี้ก้อนใหญ่? (Debt by Generation)
# ---------------------------------------------------------

# ใช้ groupby เพื่อนับจำนวนแทน crosstab
debt_gen = df.groupby(['age_group', 'debt']).size().unstack()

debt_gen.plot(kind='bar', stacked=True, figsize=(10,6))
plt.title('Insight 1: Debt by Generation')
plt.ylabel('จำนวนคน (Count)')
plt.xlabel('ช่วงวัย (Generation)')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# ---------------------------------------------------------
# Insight 2: กับดักความสุขชั่วคราว (Impulse Buying vs. Risk Level)
# สไตล์: กราฟเส้น (Line Chart) ดูแนวโน้มค่าเฉลี่ย
# ---------------------------------------------------------

# เรียงลำดับ Risk Level
risk_order = ['Low Risk', 'Medium Risk', 'High Risk']

# หาค่าเฉลี่ยของการซื้อตามใจในแต่ละกลุ่มความเสี่ยง
impulse_avg = df.groupby('risk_level')['impulse_buying'].mean().reindex(risk_order)

# พล็อตกราฟเส้น ใส่จุด (marker) ให้ดูชัดเจน
impulse_avg.plot(kind='line', marker='o', color='purple', linewidth=2, markersize=8, figsize=(8,5))
plt.title('Insight 2: แนวโน้มการซื้อตามใจ (ค่าเฉลี่ย) เทียบกับความเสี่ยง')
plt.xlabel('ระดับความเสี่ยง')
plt.ylabel('คะแนนเฉลี่ยการซื้อตามใจ (1-5)')
plt.grid(True)
plt.show()

In [ ]:
# ---------------------------------------------------------
# Insight 3: เงินเดือนเยอะแปลว่ารอดจริงหรือ? (Income vs. Risk Level)
# สไตล์: กราฟแท่งสัดส่วน 100% (100% Stacked Bar Chart)
# ---------------------------------------------------------

# สร้างกลุ่มรายได้เพื่อง่ายต่อการพล็อต
income_labels = {
    5000: '<10k',
    17500: '10k-25k',
    37500: '25k-50k',
    75000: '>50k'
}
df['income_label'] = df['income_num'].map(income_labels)
income_order = ['<10k', '10k-25k', '25k-50k', '>50k']

# 1. นับจำนวนคนแยกตาม รายได้ และ ความเสี่ยง
income_risk_count = df.groupby(['income_label', 'risk_level']).size().unstack()

# 2. แปลงเป็นเปอร์เซ็นต์ (หารด้วยผลรวมของแต่ละแถว)
income_risk_percent = income_risk_count.div(income_risk_count.sum(axis=1), axis=0)

# 3. เรียงลำดับแกน X ให้ถูกต้อง
income_risk_percent = income_risk_percent.reindex(income_order)

colors = {'Low Risk': 'green', 'Medium Risk': 'orange', 'High Risk': 'red'}
income_risk_percent.plot(kind='bar', stacked=True, color=[colors[c] for c in income_risk_percent.columns], figsize=(10,6))
plt.title('Insight 3: สัดส่วนความเสี่ยงในแต่ละช่วงรายได้ (100% Stacked)')
plt.xlabel('ช่วงรายได้')
plt.ylabel('สัดส่วน (Percentage)')
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
# ---------------------------------------------------------
# Insight 4: แผนรับมือเมื่อวิกฤตมาเยือน (Coping Strategy by Gen)
# สไตล์: กราฟแท่งจัดกลุ่ม (Grouped Bar Chart)
# ---------------------------------------------------------

coping_gen = df.groupby(['coping_strategy', 'age_group']).size().unstack()

coping_gen.plot(kind='bar', figsize=(12,6))
plt.title('Insight 4: Coping Strategy by Generation')
plt.xlabel('แนวทางการรับมือ')
plt.ylabel('จำนวนคน (Count)')
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
# ---------------------------------------------------------
# Insight 5: ความรู้คือภูมิคุ้มกันชั้นดี? (Financial Literacy vs. Risk Level)
# สไตล์: กราฟแท่งแนวนอน (Horizontal Bar Chart) ดูค่าเฉลี่ย
# ---------------------------------------------------------

# หาค่าเฉลี่ยความรู้การเงิน
lit_avg = df.groupby('risk_level')['financial_literacy'].mean().reindex(['High Risk', 'Medium Risk', 'Low Risk'])

lit_avg.plot(kind='barh', color=['red', 'orange', 'green'], figsize=(8,5))
plt.title('Insight 5: ค่าเฉลี่ยความรู้ทางการเงิน เทียบกับความเสี่ยง')
plt.xlabel('คะแนนเฉลี่ยความรู้ทางการเงิน (1-5)')
plt.ylabel('ระดับความเสี่ยง')
plt.show()